# LightFM：算法、评估与推荐

先运行 `python train.py --model lightfm --loss logistic`。本 Notebook 展示选定模型，并与 ItemCF 使用相同用户和候选规则比较。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / 'src' / 'models' / 'lightfm.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'lightfm_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model lightfm。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新训练 LightFM。')

## 1. 算法与选定参数

用户输入：性别、年龄段、职业，以及历史高分电影的类型偏好和年代偏好，共 58 个特征。
电影输入：类型、上映年代、热度、平滑平均评分和平滑高分比例，共 31 个特征。类别和数值特征加权组合成向量，ID 仅负责对齐。

训练集内部按用户时间分组约对半划分：前段构造统计特征，后段学习参数，避免目标评分参与自身统计。
验证/测试表示用完整训练集重建特征；验证和测试评分都不参与特征统计和参数学习。与原属性版本的比较也包含拟合样本数量变化。

采用 **logistic 损失**：参数拟合评分 ≥4 为 +1，1–2 分为 -1，3 分和未评分项不参与拟合。
按验证 **NDCG@10** 选参，本次比较第 5、10 轮，维度 32、种子 42、单线程。

In [2]:
display(pd.DataFrame([run['selected_parameters']]))

,loss,user_features,item_features,no_components,alpha,epochs
0,logistic,demographics+genre_preferences+decade_preferences,genres+decade+popularity+smoothed_ratings,32,0.000001,10


## 2. 与 ItemCF 比较

两种算法使用相同数据划分、评估用户和完整未评分候选目录。各指标先按用户计算再平均：
Precision@10＝命中数 /10，Recall@10＝命中数 /高分目标数，NDCG@10 同时考虑命中位置；MeanHits@10 为平均命中部数。
未评分电影不计为已知命中，但其偏好未知。LightFM 的用户向量仅由训练集学习，验证历史在测试时用于过滤已评分电影。

In [3]:
comparison = pd.read_csv(TABLE_DIR / 'comparison_metrics.csv')
comparison = comparison[comparison.algorithm.isin(['ItemCF', 'LightFM'])]
display(comparison.fillna({'loss': '-'})[['algorithm', 'loss', 'split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10', 'MeanHits@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}', 'MeanHits@10': '{:.3f}'}))

,algorithm,loss,split,users,Precision@10,Recall@10,NDCG@10,MeanHits@10
0,ItemCF,-,validation,5776,5.97%,7.89%,0.0844,0.597
1,ItemCF,-,test,5765,5.64%,7.12%,0.0773,0.564
2,LightFM,logistic,validation,5776,1.94%,1.80%,0.0230,0.194
3,LightFM,logistic,test,5765,1.96%,1.84%,0.0237,0.196


## 3. 按用户比较推荐电影

修改 `USER_ID` 后运行下面的单元格。每种算法推荐 5 部历史未评分电影，展示标题与类型。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = []
for algorithm in ['itemcf', 'lightfm']:
    result = recommend_for_user(USER_ID, stage='test', k=TOP_K, algorithm=algorithm)
    result.insert(0, 'algorithm', algorithm)
    recommendations.append(result[['algorithm', 'rank', 'title', 'genres']])
display(pd.concat(recommendations, ignore_index=True))

,algorithm,rank,title,genres
0,itemcf,1,Raiders of the Lost Ark (1981),Action|Adventure
1,itemcf,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,itemcf,3,"Shawshank Redemption, The (1994)",Drama
3,itemcf,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,itemcf,5,Pulp Fiction (1994),Crime|Drama
5,lightfm,1,Citizen Kane (1941),Drama
6,lightfm,2,It's a Wonderful Life (1946),Drama
7,lightfm,3,"Maltese Falcon, The (1941)",Film-Noir|Mystery
8,lightfm,4,North by Northwest (1959),Drama|Thriller
9,lightfm,5,12 Angry Men (1957),Drama


算法参考：Kula (2015), [Metadata Embeddings for User and Item Cold-start Recommendations](https://arxiv.org/abs/1507.08439)。训练使用[官方 LightFM 实现](https://making.lyst.com/lightfm/docs/lightfm.html)。